# M3L2 E05 - OpenAIEmbeddings: convertir texto en vectores

## Un solo concepto

Un **embedding** es una lista de numeros que representa el significado de un texto.

Textos con significado similar tienen vectores similares (cercanos en el espacio).
Textos con significado diferente tienen vectores lejanos.

Este es el fundamento de todo el sistema RAG:
el retriever busca documentos **cercanos** al vector de la pregunta.

## No necesita FAISS — solo OpenAI API key

Vamos a calcular vectores y comparar similitudes directamente, sin base de datos.


In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import OpenAIEmbeddings
import math


## Por que necesitamos embeddings: el corazon del RAG (Lecture M3L2 - Seccion 13)

La lecture M3L2 describe el flujo de indexacion asi:

```text
INGESTION (se hace una vez):
  Documentos
     |
     v
  Chunks (fragmentos de texto)
     |
     v
  Embeddings (vectores numericos)
     |
     v
  Vector Store (base de datos de vectores)
     |
     v
  Retriever (interfaz de busqueda)
```

**El problema que resuelven los embeddings**:

En M3L1 y en el script legacy de M3L2, haciamos busqueda "naive":
manda TODO el contexto al modelo sin filtrar.

Con 5 documentos no se nota. Con 5.000 documentos, el costo de tokens
puede ser 1.000 veces mayor y la calidad de la respuesta puede bajar
porque el modelo tiene mas ruido para filtrar.

**Los embeddings permiten**:

```text
Pregunta: "Cuantos dias de vacaciones tengo?"
     |
     v
  Embedding de la pregunta: [0.12, -0.45, 0.89, ...]
     |
     v
  Buscar vectores SIMILARES en el Vector Store
     |
     v
  Solo traer los 2-3 documentos mas relevantes
  (los que tienen vectores mas cercanos al de la pregunta)
```

| Sin embeddings (M3L1 / script legacy) | Con embeddings (FAISS + Retriever) |
|---|---|
| Manda TODOS los docs siempre | Solo manda los k mas relevantes |
| Costo O(n) en tokens | Costo O(k) en tokens, k << n |
| Mas ruido para el modelo | Contexto filtrado y relevante |
| No escala con muchos docs | Escala a millones de documentos |


## Paso 1: crear el modelo de embeddings

`OpenAIEmbeddings` convierte texto en vectores numericos usando el modelo `text-embedding-ada-002`.

Tiene dos metodos principales:
- `.embed_query(texto)`: embeder una sola pregunta/consulta
- `.embed_documents([texto1, texto2, ...])`: embeder una lista de documentos


In [ ]:
# TODO 1: crear el objeto OpenAIEmbeddings
# embeddings = OpenAIEmbeddings()
embeddings = None  # reemplazar

print(f"Tipo: {type(embeddings).__name__ if embeddings else 'TODO no completado'}")


## Paso 2: embeder un texto y ver su forma

El vector de un texto es una lista de 1536 numeros (para el modelo ada-002).
Cada numero representa una dimension del espacio semantico.


In [ ]:
# TODO 2: embeder el texto "vacaciones" con embeddings.embed_query()
# vector = embeddings.embed_query("vacaciones")

# Descomentar despues de completar TODO 1 y 2:
# print(f"Tipo del vector: {type(vector).__name__}")
# print(f"Longitud del vector: {len(vector)} dimensiones")
# print(f"Primeros 5 valores: {[round(x, 4) for x in vector[:5]]}")
# print()
# print("Este vector numerco representa el 'significado' de la palabra 'vacaciones'")


## Paso 3: la similitud del coseno

Para comparar dos vectores usamos la **similitud del coseno**.
Devuelve un valor entre -1 y 1:
- `1.0` = textos identicos o muy similares
- `0.0` = textos sin relacion
- `-1.0` = textos opuestos

Esta funcion ya esta dada, no es un TODO.


In [ ]:
def cosine_similarity(v1: list, v2: list) -> float:
    """Calcula la similitud del coseno entre dos vectores."""
    dot = sum(a * b for a, b in zip(v1, v2))
    norm1 = math.sqrt(sum(a * a for a in v1))
    norm2 = math.sqrt(sum(b * b for b in v2))
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)

print("Funcion cosine_similarity lista.")


## TODO 3: comparar textos similares vs diferentes

Embede estos 4 textos y calcula la similitud entre pares.

Hipotesis:
- Los dos textos sobre vacaciones deben tener similitud ALTA.
- El texto sobre vacaciones vs el texto sobre futbol debe tener similitud BAJA.


In [ ]:
textos = [
    "Cuantos dias de vacaciones tengo?",
    "Politica de licencia y dias libres de la empresa",
    "El partido de futbol fue emocionante",
    "Quiero saber sobre los beneficios laborales",
]

# TODO 3: embeder todos los textos con embeddings.embed_documents(textos)
# vectores = embeddings.embed_documents(textos)

# Descomentar para ver la comparacion:
# pares = [
#     (0, 1, "Vacaciones vs Licencia (similar)"),
#     (0, 2, "Vacaciones vs Futbol (diferente)"),
#     (0, 3, "Vacaciones vs Beneficios (relacionado)"),
#     (1, 3, "Licencia vs Beneficios (relacionado)"),
# ]
# print("Similitud del coseno entre textos:")
# for i, j, label in pares:
#     sim = cosine_similarity(vectores[i], vectores[j])
#     print(f"  {label}: {sim:.4f}")


In [ ]:
def run_checks():
    assert embeddings is not None, "TODO 1: embeddings es None"
    v = embeddings.embed_query("test")
    assert isinstance(v, list) and len(v) > 100, "El vector debe tener muchas dimensiones"
    # Similitud entre el mismo texto debe ser ~1.0
    v1 = embeddings.embed_query("vacaciones")
    v2 = embeddings.embed_query("vacaciones")
    sim = cosine_similarity(v1, v2)
    assert sim > 0.99, f"Mismo texto debe tener similitud ~1.0, es {sim:.4f}"
    # Textos distintos deben tener similitud menor
    v3 = embeddings.embed_query("futbol")
    sim2 = cosine_similarity(v1, v3)
    assert sim2 < sim, "Textos diferentes deben tener menor similitud"
    print("M3L2 E05 checks passed")

run_checks()


## Cierre

| Concepto | Que es |
|---|---|
| Embedding | Lista de numeros que representa el significado de un texto |
| `embed_query()` | Embeder una pregunta o consulta |
| `embed_documents()` | Embeder una lista de documentos |
| Similitud del coseno | Mide que tan "cercanos" son dos vectores |
| Por que importa | El retriever busca documentos cuyo vector sea similar al de la pregunta |

**Siguiente**: E06 muestra como FAISS usa estos vectores para hacer busquedas eficientes.
